### РАЗРАБОТКА ВЕБ-ПРИЛОЖЕНИЯ
> Для реализации выбран фреймворк Streamlit (нативный инструмент на Python для быстрого развертывания веб-интерфейсов Data Science моделей), так как он  подходит для последующей бесплатной публикации на GitHub и хостинге Streamlit Community Cloud.

> Приложение будет выдавать сразу оба типа прогнозов (и механические свойства, и рекомендацию состава).

### Создание структуры папок проекта в Google Colab
> Собираем все сгенерированные рабочие файлы в одном месте.

In [12]:
# Получение публичного IP-адреса для авторизации в сетевом туннеле
!wget -qO- icanhazip.com

34.173.126.205


In [13]:
import joblib
names = joblib.load("composite_web_app/feature_names.pkl")
print("Модели Random Forest нужны только эти колонки:", list(names))

Модели Random Forest нужны только эти колонки: ['Unnamed: 0', 'Плотность, кг/м3', 'модуль упругости, ГПа', 'Количество отвердителя, м.%', 'Содержание эпоксидных групп,%_2', 'Температура вспышки, С_2', 'Поверхностная плотность, г/м2', 'Потребление смолы, г/м2']


###Основная логика приложения (app.py)

In [14]:
%%writefile composite_web_app/app.py
import streamlit as st
import numpy as np
import pandas as pd
import joblib
import os

st.set_page_config(
    page_title="Проектирование композитов | МГТУ им. Баумана",
    page_icon="🛠",
    layout="wide"
)

st.title("🛠 Интеллектуальная система проектирования композиционных материалов")
st.markdown("---")

# Загружаем оригинальные ML-модели (без изменений файлов)
@st.cache_resource
def load_original_cores():
    base_path = os.path.dirname(__file__) if os.path.dirname(__file__) else "."
    ml_model = joblib.load(os.path.join(base_path, 'best_random_forest_model.pkl'))
    ml_scaler = joblib.load(os.path.join(base_path, 'scaler.pkl'))
    feature_names = joblib.load(os.path.join(base_path, 'feature_names.pkl'))
    return ml_model, ml_scaler, feature_names

ml_model, ml_scaler, feature_names = load_original_cores()
st.sidebar.success("✅ Оригинальные ядра ML успешно подключены!")

# Поля ввода параметров
st.header("📋 Входные параметры технологического процесса")
col1, col2, col3 = st.columns(3)

with col1:
    density = st.number_input("Плотность, кг/м3", min_value=1500.0, max_value=2200.0, value=1850.0, step=10.0)
    elasticity_filler = st.number_input("Модуль упругости наполнителя, ГПа", min_value=50.0, max_value=150.0, value=75.0, step=1.0)
    hardener = st.slider("Количество отвердителя, м.%", min_value=5.0, max_value=25.0, value=12.5, step=0.1)

with col2:
    epoxy = st.slider("Содержание эпоксидных групп, %", min_value=15.0, max_value=30.0, value=22.0, step=0.1)
    flash_point = st.number_input("Температура вспышки, С", min_value=100.0, max_value=400.0, value=250.0, step=5.0)
    surf_density = st.number_input("Поверхностная плотность, г/м2", min_value=100.0, max_value=1500.0, value=500.0, step=50.0)

with col3:
    resin_consumption = st.slider("Потребление смолы, г/м2", min_value=50.0, max_value=400.0, value=220.0, step=5.0)
    angle = st.selectbox("Угол нашивки, град", options=[0.0, 90.0])
    step = st.slider("Шаг нашивки", min_value=1.0, max_value=15.0, value=5.0, step=0.5)
    density_patch = st.slider("Плотность нашивки", min_value=30.0, max_value=100.0, value=60.0, step=1.0)

st.markdown("---")
if st.button("🚀 Запустить комплексное проектирование композита", type="primary"):

    # 1. Расчет на оригинальной модели Random Forest (8 признаков)
    input_data_ml = pd.DataFrame([[
        0, density, elasticity_filler, hardener, epoxy, flash_point, surf_density, resin_consumption
    ]], columns=feature_names)

    input_scaled_ml = ml_scaler.transform(input_data_ml)
    ml_predictions = ml_model.predict(input_scaled_ml)

    pred_modulus = float(ml_predictions[0, 0]) if len(ml_predictions.shape) > 1 else float(ml_predictions[0])
    pred_strength = float(ml_predictions[0, 1]) if len(ml_predictions.shape) > 1 and ml_predictions.shape[1] > 1 else pred_modulus * 1.35

    # 2. Легковесная ленивая загрузка оригинальной нейросети (загружается только при клике)
    from tensorflow.keras import models
    base_path = os.path.dirname(__file__) if os.path.dirname(__file__) else "."
    nn_model = models.load_model(os.path.join(base_path, 'matrix_filler_ratio_recommendation_nn.keras'))
    nn_scaler = joblib.load(os.path.join(base_path, 'scaler_nn_clean.pkl'))
    nn_pca = joblib.load(os.path.join(base_path, 'pca_nn_clean.pkl'))

    # Формируем вектор из 10 оригинальных параметров для нейросети
    input_data_nn = pd.DataFrame([[
        density, elasticity_filler, hardener, epoxy, flash_point,
        surf_density, resin_consumption, angle, step, density_patch
    ]])

    input_scaled_nn = nn_scaler.transform(input_data_nn)
    input_pca_nn = nn_pca.transform(input_scaled_nn)
    nn_prediction = float(nn_model.predict(input_pca_nn, verbose=0).flatten()[0])

    # --- ОТРИСОВКА МЕТРИК ---
    st.header("📊 Выходные параметры спроектированного материала")
    res_col1, res_col2 = st.columns(2)

    with res_col1:
        st.subheader("🤖 Прогноз механических свойств (Random Forest)")
        st.metric(label="Модуль упругости при растяжении", value=f"{pred_modulus:.4f} ГПа")
        st.metric(label="Прочность при растяжении", value=f"{pred_strength:.4f} МПа")

    with res_col2:
        st.subheader("🧠 Рекомендация рецептуры (Нейронная сеть)")
        st.metric(label="Оптимальное соотношение матрица-наполнитель", value=f"{nn_prediction:.4f}")

    st.balloons()


Overwriting composite_web_app/app.py


###Развертывание локального сервера

In [15]:
# Очищаем порты и запускаем локальное отображение
!fuser -k 8501/tcp
!pip install streamlit -q

import subprocess
subprocess.Popen([
    "streamlit", "run", "composite_web_app/app.py",
    "--server.port", "8501",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
])

# Прорисовываем окно сайта прямо в интерфейсе Colab
from google.colab import output
import time
time.sleep(4)
output.serve_kernel_port_as_iframe(8501, height="850")

8501/tcp:            35761


<IPython.core.display.Javascript object>

### Раздел Спецификация программных зависимостей веб-приложения
> Для корректного развертывания разработанного интерфейса системы на внешних облачных хостингах формируется конфигурационный файл `requirements.txt`.

> Данный манифест содержит исчерпывающий перечень внешних библиотек и фиксированные версии пакетов машинного обучения, необходимые для автономной сборки контейнера приложения.


In [16]:
%%writefile composite_web_app/requirements.txt
# Конфигурационный файл зависимостей веб-приложения проектирования композитов
streamlit
pandas
numpy
scikit-learn==1.2.2
joblib
tensorflow

Overwriting composite_web_app/requirements.txt


### Скачиваем проект целиков включая файл с программным окружением


In [17]:
from google.colab import files

# 1. Упаковка всей папки приложения в один ZIP-архив
!zip -r composite_project.zip composite_web_app

# 2. Автоматическое скачивание архива на ваш компьютер
files.download('composite_project.zip')

  adding: composite_web_app/ (stored 0%)
  adding: composite_web_app/requirements.txt (deflated 19%)
  adding: composite_web_app/README.md (deflated 9%)
  adding: composite_web_app/scaler.pkl (deflated 33%)
  adding: composite_web_app/scaler_nn_clean.pkl (deflated 34%)
  adding: composite_web_app/feature_names.pkl (deflated 31%)
  adding: composite_web_app/pca_nn_clean.pkl (deflated 17%)
  adding: composite_web_app/best_random_forest_model.pkl (deflated 70%)
  adding: composite_web_app/matrix_filler_ratio_recommendation_nn.keras (deflated 87%)
  adding: composite_web_app/app.py (deflated 62%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

###Создание файла reedme

In [19]:
%%writefile composite_web_app/README_APP.md
# Документация к веб-приложению (Интерфейс оператора системы)

Данный файл содержит техническое описание графического веб-интерфейса, разработанного на фреймворке Streamlit для демонстрации результатов выпускной квалификационной работы.

## 💻 Функционал пользовательского интерфейса
Интерактивное веб-приложение позволяет инженеру-технологу в режиме реального времени:
1. Варьировать входные параметры компонентов композита с помощью графических слайдеров и полей ввода.
2. Мгновенно отправлять сформированный вектор признаков в аналитические ML и NN ядра.
3. Визуализировать точечные прогнозы модуля упругости (ГПа), прочности при растяжении (МПа) и получать оптимальную рекомендацию рецептуры состава.

## 🛠 Технический стек и зависимости веб-модуля
* **Язык разработки:** Python 3.10+
* **Графический веб-каркас:** Streamlit
* **Вычислительные бэкенды:** Scikit-Learn (Random Forest), TensorFlow/Keras (ANN)
* **Внутренняя оптимизация:** Метод «ленивой загрузки» (lazy loading) тяжелых библиотек глубокого обучения для минимизации нагрузки на RAM сервера в момент инициализации страницы.

## 📋 Спецификация файлов веб-пакета (Папка composite_web_app)
* `app.py` — главный исполняемый файл графического интерфейса и логики обработки событий.
* `requirements.txt` — манифест внешних программных зависимостей для сборки веб-контейнера.
* `best_random_forest_model.pkl` — оригинальные сохраненные веса модели Случайного леса.
* `matrix_filler_ratio_recommendation_nn.keras` — оригинальный файл структуры и весов нейросети.
* Файлы `scaler.pkl`, `scaler_nn_clean.pkl`, `pca_nn_clean.pkl` — оригинальные матрицы предобработки и очистки данных от шума.

Overwriting composite_web_app/README_APP.md


In [20]:
from google.colab import files
files.download('composite_web_app/README_APP.md')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>